# Legal RAG Single-Case Run

Run one federal sentencing case through the legal RAG baseline. This notebook is written to work both locally and on Databricks.

Prerequisites:
- The legal index has already been chunked and uploaded.
- The strict sentencing table is available through Spark.
- Azure Search and Azure OpenAI settings are available in `.env` or the active environment.
- If you exported USSG Doc Intelligence text, make sure `LEGAL_USSG_DOCINTEL_TEXT_ROOT` points at that cache before you rebuild the index.

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd

cwd = Path.cwd().resolve()
project_root = next((candidate for candidate in [cwd, *cwd.parents] if (candidate / "pyproject.toml").exists()), cwd)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from baselines.legal_rag import (
    fetch_case_record,
    load_config,
    resolve_execution_env,
    resolve_legal_prompts_dir,
    resolve_model_name,
    resolve_spark_session,
    resolve_strict_table,
    run_single_case_prediction,
    score_prediction,
)

c:\Users\sonutka\main\2-Envs\tam-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
execution_env = resolve_execution_env()
spark = resolve_spark_session(app_name="legal-rag-single-case")
config = load_config()
strict_table = resolve_strict_table()
prompts_dir = resolve_legal_prompts_dir()
model_name = resolve_model_name()

summary = {
    "execution_env": execution_env,
    "index_name": config.index_name,
    "search_service": config.search_service.name,
    "ussg_root": str(config.ussg_root),
    "title18_root": str(config.title18_root),
    "ussg_docintel_text_root": None if config.ussg_docintel_text_root is None else str(config.ussg_docintel_text_root),
    "strict_table": strict_table,
    "prompts_dir": str(prompts_dir),
    "model_name": model_name,
}
summary

{'execution_env': 'local',
 'index_name': 'legal-rag-baseline',
 'search_service': 'service_1',
 'ussg_root': 'C:\\Volumes\\usdo_aa_catalog\\research_tam_datasets\\federal_sentencing\\legal_manuals\\ussg',
 'title18_root': 'C:\\Volumes\\usdo_aa_catalog\\research_tam_datasets\\federal_sentencing\\legal_manuals\\usc_title18',
 'ussg_docintel_text_root': 'C:\\Volumes\\usdo_aa_catalog\\research_tam_datasets\\federal_sentencing\\legal_manuals\\_docintel_text\\ussg',
 'strict_table': 'usdo_aa_catalog.research_tam_datasets.federal_sentencing_strict_docintel_script',
 'prompts_dir': 'C:\\Users\\sonutka\\main\\1-Projects\\manual_deterministic_executor\\baselines\\legal_rag\\prompts',
 'model_name': 'openai:gpt-5'}

## Choose a Case

Set `year` and `docket_id` below. The notebook uses a concrete default docket so you can run all cells immediately.

In [3]:
year = 2024
docket_id = 18419431
top_k = 8
llm_max_attempts = 2

strict_df = spark.table(strict_table)

preview_columns = [
    "year",
    "docket_id",
    "government_sm_doc_id",
    "government_sm_short_description",
    "offense_level_total",
    "criminal_history_category",
    "guidelines_low_months",
    "guidelines_high_months",
]
available_preview_columns = [column_name for column_name in preview_columns if column_name in strict_df.columns]
preview_df = strict_df.filter(f"year = {int(year)}").select(*available_preview_columns).orderBy("year", "docket_id").limit(20).toPandas()
preview_df

,year,docket_id,government_sm_doc_id,government_sm_short_description,offense_level_total,criminal_history_category,guidelines_low_months,guidelines_high_months
0,2024,18419431,407842611,Sentencing Memorandum,21,I,37,46
1,2024,18421869,398568130,Sentencing Memorandum,33,VI,235,240
2,2024,59293248,397256119,Sentencing Memorandum,15,I,18,24
3,2024,59674994,414307776,Sentencing Memorandum,None,None,None,None
4,2024,59767658,421324282,Sentencing Memorandum,17,I,24,30
5,2024,59767659,419852955,Sentencing Memorandum,10,I,6,12
6,2024,59940867,422075418,Sentencing Memorandum,10,I,6,12
7,2024,60070206,415848988,Sentencing Memorandum,6,I,0,6
8,2024,60113400,391441341,Sentencing Memorandum,10,II,8,14
9,2024,60372325,407557244,Sentencing Memorandum,32,VI,210,262


In [4]:
case_record = fetch_case_record(
    spark=spark,
    table_name=strict_table,
    docket_id=docket_id,
    year=year,
)

case_preview = {
    "year": case_record.get("year"),
    "docket_id": case_record.get("docket_id"),
    "government_sm_doc_id": case_record.get("government_sm_doc_id"),
    "government_sm_short_description": case_record.get("government_sm_short_description"),
    "expected_offense_level_total": case_record.get("expected_offense_level_total"),
    "expected_criminal_history_category": case_record.get("expected_criminal_history_category"),
    "expected_guidelines_low_months": case_record.get("expected_guidelines_low_months"),
    "expected_guidelines_high_months": case_record.get("expected_guidelines_high_months"),
    "charges_or_offense": case_record.get("charges_or_offense"),
}

print(json.dumps(case_preview, indent=2))
print()
print(case_record["case_summary"][:4000])

{
  "year": 2024,
  "docket_id": 18419431,
  "government_sm_doc_id": 407842611,
  "government_sm_short_description": "Sentencing Memorandum",
  "expected_offense_level_total": "21",
  "expected_criminal_history_category": "I",
  "expected_guidelines_low_months": "37",
  "expected_guidelines_high_months": "46",
  "charges_or_offense": [
    "Conspiracy (18 U.S.C. \u00a7 371) to violate the International Emergency Economic Powers Act (IEEPA)",
    "Willful violations of IEEPA (50 U.S.C. \u00a7 1705; 31 C.F.R. \u00a7\u00a7 560.204, 560.206; Exec. Order No. 13,876; and 18 U.S.C. \u00a7 2)"
  ]
}

Defendant A pleaded guilty to a conspiracy and to willful violations of U.S. sanctions laws. From about December 2018 through December 2019, Defendant A and others collected a religious tax (khums) and other U.S. currency in the United States—much of it solicited through an online "Yemen campaign"—and arranged to move that money to Iran, including to the office of the Supreme Leader, without an OF

In [5]:
result = run_single_case_prediction(
    config=config,
    summary_text=case_record["case_summary"],
    model_name=model_name,
    prompts_dir=prompts_dir,
    top_k=top_k,
    llm_max_attempts=llm_max_attempts,
    source_year=case_record.get("year"),
)

metrics = score_prediction(result["prediction"], case_record)
output = {
    "prediction": result["prediction"],
    "metrics": metrics,
}
print(json.dumps(output, indent=2))

{
  "prediction": {
    "predicted_offense_level_total": "26",
    "predicted_criminal_history_category": null,
    "predicted_guidelines_low_months": null,
    "predicted_guidelines_high_months": null,
    "confidence": 0.62,
    "rationale": "The offense is an IEEPA sanctions-violations scheme moving U.S. dollars to Iran, a designated state sponsor of terrorism. Under USSG \u00a72M5.1 (which governs 50 U.S.C. \u00a71705 offenses), the base offense level is 26 when the conduct involves financial transactions with a country supporting international terrorism. No additional adjustments are clearly supported by the provided context, and criminal history is not provided.",
    "supporting_evidence": [
      "USSG \u00a72M5.1 applies to 50 U.S.C. \u00a71705 offenses and sets a higher base level for financial transactions with countries supporting international terrorism; Application Note defines such countries (Iran qualifies).",
      "Case facts: willful, unlicensed transfer of about $90

In [6]:
retrieved_preview = pd.DataFrame(result["retrieved_chunks"])
retrieved_preview[[
    "source_type",
    "source_year",
    "chunk_title",
    "citation",
    "score",
]]

,source_type,source_year,chunk_title,citation,score
0,usc_title18,2024,§2339C. Prohibitions against the financing of ...,18 U.S.C. § 2339C,0.023472
1,ussg,2024,§ 2339A; or (ii) the conduct involved harborin...,§ 2339A,0.022569
2,usc_title18,2024,§2339B. Providing material support or resource...,18 U.S.C. § 2339B,0.018080
3,usc_title18,2024,§2332d. Financial transactions,18 U.S.C. § 2332d,0.016667
4,ussg,2024,"§ 3561. Under current law, if the court finds ...",§ 3561.,0.016667
5,ussg,2024,"§ 924(e). Under 18 U.S.C. § 924(e)(1), a defen...",§ 924,0.016393
6,ussg,2024,§2M5.1. Evasion of Export Controls; Financial ...,§2M5.1.,0.016393
7,usc_title18,2024,§247. Damage to religious property; obstructio...,18 U.S.C. § 247,0.016129
